# KitchenHygiene v3: train a replacement head detector

Choose **Runtime > Change runtime type > T4 GPU** when available. Run cells in order. Upload **only KitchenHygiene-heads-training.zip** at the file prompt. The original KitchenHygiene.v3i.yolov8.zip also works, but needs additional preparation time. No project ZIP or API key is required.

This experiment trains hairnet/no_hairnet only, for direct integration with the existing app. It does not train apron, glove or pest detection. The prepared data contains **1,603 train / 232 validation / 298 test images**. The original export contains 9,304 mixed-domain images; kitchen and office CCTV are retained in train, with source-family/dHash components kept together. Unknown-origin augmentations and images lacking head labels are excluded. The data is **CC BY 4.0**, by Kitchen Hygiene: https://universe.roboflow.com/kitchen-hygiene-efuu5/kitchenhygiene/dataset/3 . Retain attribution files.

This is not an independent college-kitchen benchmark. New test scores are not directly comparable to the old 19.1% mAP50 because the test set differs. No improvement is claimed before training and evaluation. Colab runtime storage is temporary; download the result ZIP promptly.


In [ ]:
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','ultralytics==8.4.215','Pillow<13','PyYAML>=6,<7'])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun this cell'
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
stage=Path('/content/kitchen-training-code');stage.mkdir(exist_ok=True)
(stage/'train.py').write_text('"""Validate a YOLO dataset, fine-tune a detector, and evaluate held-out data."""\nimport argparse, hashlib, json, math, csv\nfrom pathlib import Path\n\ndef audit(config):\n    import yaml\n    config=Path(config).resolve()\n    data=yaml.safe_load(config.read_text())\n    root=Path(data.get(\'path\',config.parent))\n    if not root.is_absolute(): root=(config.parent/root).resolve()\n    names=data[\'names\']\n    names=list(names.values()) if isinstance(names,dict) else names\n    if sorted(names)!=[\'hairnet\',\'no_hairnet\']:\n        raise ValueError(\'Dataset must have exactly hairnet and no_hairnet classes. Remap labels explicitly first.\')\n    seen={}; report={\'config\':str(config),\'names\':names,\'splits\':{}}\n    for split in (\'train\',\'val\',\'test\'):\n        folder=root/str(data.get(split,\'\'))\n        if not data.get(split) or not folder.is_dir():\n            raise ValueError(f\'{split} must point to an existing image directory.\')\n        images=sorted(p for p in folder.rglob(\'*\') if p.suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'))\n        if not images: raise ValueError(f\'{split} is empty\')\n        counts=[0]*len(names)\n        for image in images:\n            digest=hashlib.sha256(image.read_bytes()).hexdigest()\n            if digest in seen and seen[digest]!=split:\n                raise ValueError(f\'Exact duplicate across splits: {image}\')\n            seen[digest]=split\n            parts=list(image.parts)\n            if \'images\' not in parts: raise ValueError(\'Expected images/ and labels/ parallel folders.\')\n            parts[len(parts)-1-parts[::-1].index(\'images\')]=\'labels\'\n            label=Path(*parts).with_suffix(\'.txt\')\n            if not label.exists(): raise ValueError(f\'Missing label: {label}; use empty files for verified negatives.\')\n            for line in label.read_text().splitlines():\n                values=line.split()\n                if len(values)!=5: raise ValueError(f\'Invalid YOLO label: {label}\')\n                cls=int(values[0]); x,y,w,h=map(float,values[1:])\n                if not 0<=cls<len(names) or not all(math.isfinite(v) for v in (x,y,w,h)) or not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1) or x-w/2 < -0.001 or x+w/2 > 1.001 or y-h/2 < -0.001 or y+h/2 > 1.001:\n                    raise ValueError(f\'Out-of-range annotation: {label}\')\n                counts[cls]+=1\n        if not all(counts): raise ValueError(f\'{split} must contain both classes.\')\n        report[\'splits\'][split]={\'images\':len(images),\'objects\':dict(zip(names,counts))}\n    return report\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--data\',required=True)\n    p.add_argument(\'--audit-only\',action=\'store_true\')\n    p.add_argument(\'--epochs\',type=int,default=50)\n    p.add_argument(\'--device\',default=\'cpu\')\n    p.add_argument(\'--batch\',type=int,default=8)\n    p.add_argument(\'--imgsz\',type=int,default=640)\n    p.add_argument(\'--output\',default=\'runs/hairnet\')\n    p.add_argument(\'--freeze\',type=int,default=0)\n    p.add_argument(\'--threads\',type=int,default=4)\n    p.add_argument(\'--patience\',type=int,default=7)\n    p.add_argument(\'--class-power\',type=float,default=0.0,help=\'Ultralytics cls_pw; supported by the pinned notebook version\')\n    args=p.parse_args()\n    report=audit(args.data)\n    output=Path(args.output).resolve(); output.mkdir(parents=True,exist_ok=True)\n    (output/\'dataset_audit.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n    if args.audit_only: return\n    from ultralytics import YOLO\n    import torch\n    torch.set_num_threads(args.threads)\n    model=YOLO(\'yolo11n.pt\')\n    weighting={\'cls_pw\':args.class_power} if args.class_power else {}\n    result=model.train(data=str(Path(args.data).resolve()),epochs=args.epochs,device=args.device,\n        batch=args.batch,imgsz=args.imgsz,workers=0,seed=42,deterministic=True,\n        project=str(output),name=\'training\',exist_ok=False,patience=args.patience,freeze=args.freeze or None,**weighting)\n    best=Path(result.save_dir)/\'weights/best.pt\'\n    metrics=YOLO(str(best)).val(data=str(Path(args.data).resolve()),split=\'test\',device=args.device,imgsz=args.imgsz,batch=args.batch,workers=0,\n        project=str(output),name=\'held_out_test\',plots=True)\n    per_class={}\n    for i,cls in enumerate(metrics.box.ap_class_index):\n        per_class[report[\'names\'][int(cls)]]={\'precision\':float(metrics.box.p[i]),\'recall\':float(metrics.box.r[i]),\'AP50\':float(metrics.box.ap50[i]),\'AP50_95\':float(metrics.box.ap[i])}\n    history=list(csv.DictReader((Path(result.save_dir)/\'results.csv\').open()))\n    card={\'status\':\'trained\',\'weights\':str(best),\'dataset\':report,\'test_metrics\':metrics.results_dict,\'per_class_test_metrics\':per_class,\n        \'epochs_requested\':args.epochs,\'epochs_completed\':len(history),\'base_model\':\'yolo11n.pt\',\'imgsz\':args.imgsz,\'freeze\':args.freeze,\'patience\':args.patience,\'class_power\':args.class_power,\n        \'limitations\':[\'Public data may not represent your kitchen\',\'No detection means uncertain, not absent\',\'Exact duplicate audit does not detect near duplicates; split by source session before training\']}\n    (output/\'model_card.json\').write_text(json.dumps(card,indent=2,default=str))\n    print(\'Training and held-out evaluation complete:\',best)\n\nif __name__==\'__main__\': main()\n')
(stage/'prepare_kitchenhygiene.py').write_text('"""Prepare head-only KitchenHygiene v3 data with conservative group isolation."""\nimport argparse,hashlib,json,math,re\nfrom collections import Counter,defaultdict\nfrom io import BytesIO\nfrom pathlib import Path\nfrom zipfile import ZipFile\nfrom PIL import Image,ImageOps\nimport yaml\n\nOFFICE=(\'haiiiir\',\'hair03\',\'hair05\',\'hair1\',\'hairnet_102\',\'hairnet_104\',\'hairnet_ver\',\'hairr\',\'hairrr1\',\'mask_data_102\')\n\ndef source_family(filename):\n    s=Path(filename).name.split(\'.rf.\')[0]\n    if s.startswith(\'augmented_image\'):return None\n    if s.startswith(\'Frame_\'):return \'kitchen_cctv\'\n    if s.startswith(OFFICE):return \'office_camera\'\n    if s.startswith(\'video_BC\'):return \'food_production_media\'\n    if \'_mp4\' in s:return \'video:\'+s.split(\'_mp4\')[0]\n    s=re.sub(r\'_(jpg|jpeg|png)$\',\'\',s,flags=re.I)\n    return \'unverified_source:\'+s\n\ndef dhash(raw):\n    im=ImageOps.exif_transpose(Image.open(BytesIO(raw))).convert(\'L\')\n    hashes=[]\n    for view in (im,ImageOps.mirror(im)):\n        small=view.resize((9,8));pixels=list(small.tobytes());bits=0\n        for y in range(8):\n            for x in range(8):bits=(bits<<1)|int(pixels[y*9+x]>pixels[y*9+x+1])\n        hashes.append(bits)\n    return min(hashes)\n\ndef prepare(archive,destination):\n    dest=Path(destination).resolve()\n    if dest.exists() and any(dest.iterdir()):raise ValueError(\'Destination must be new or empty; existing data was preserved.\')\n    with ZipFile(archive) as z:\n        configs=[n for n in z.namelist() if Path(n).name==\'data.yaml\']\n        if len(configs)!=1:raise ValueError(\'Expected exactly one dataset data.yaml.\')\n        original=yaml.safe_load(z.read(configs[0]));names=original[\'names\']\n        if isinstance(names,dict):names=[names[i] for i in sorted(names)]\n        target={names.index(\'hairnet\'):0,names.index(\'no_hairnet\'):1}\n        records=[];total=0;excluded=Counter();polygon_lines=0\n        image_files=sorted(n for n in z.namelist() if \'/images/\' in n and Path(n).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'))\n        for n in image_files:\n            total+=1;label=n.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            try:lines=z.read(label).decode().splitlines()\n            except KeyError:raise ValueError(f\'Missing annotation: {label}\') from None\n            selected=[];counts=[0,0]\n            for line in lines:\n                fields=line.split();cls=int(fields[0])\n                if len(fields)!=5:\n                    if len(fields)<7 or len(fields)%2!=1:raise ValueError(f\'Invalid label structure: {label}\')\n                    coordinates=list(map(float,fields[1:]))\n                    if not all(math.isfinite(v) and 0<=v<=1 for v in coordinates):raise ValueError(f\'Invalid polygon: {label}\')\n                    polygon_lines+=1\n                    if cls in target:raise ValueError(\'A head polygon needs explicit conversion; preparation stopped.\')\n                    continue\n                if cls not in target:continue\n                x,y,w,h=map(float,fields[1:])\n                if not all(math.isfinite(v) for v in (x,y,w,h)) or not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1) or min(x-w/2,y-h/2)<-.001 or max(x+w/2,y+h/2)>1.001:\n                    raise ValueError(f\'Invalid head box: {label}\')\n                new=target[cls];selected.append(f\'{new} {x:.8f} {y:.8f} {w:.8f} {h:.8f}\');counts[new]+=1\n            if not selected:excluded[\'no_head_labels\']+=1;continue\n            family=source_family(n)\n            if family is None:excluded[\'unknown_augmented_origin\']+=1;continue\n            raw=z.read(n)\n            records.append(dict(member=n,source_split=n.split(\'/\')[0],family=family,sha256=hashlib.sha256(raw).hexdigest(),dhash=dhash(raw),labels=selected,counts=counts))\n        if not records:raise ValueError(\'No usable head-labeled images found.\')\n        # Union by source family, exact image identity and a conservative visual near-duplicate heuristic.\n        parent=list(range(len(records)))\n        def find(i):\n            while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]\n            return i\n        def union(a,b):\n            a,b=find(a),find(b)\n            if a!=b:parent[b]=a\n        families={};digests={};visual=[];exact_removed=set();near_pairs=0\n        for i,r in enumerate(records):\n            if r[\'family\'] in families:union(i,families[r[\'family\']])\n            else:families[r[\'family\']]=i\n            if r[\'sha256\'] in digests:\n                previous=digests[r[\'sha256\']]\n                if records[previous][\'labels\']!=r[\'labels\']:raise ValueError(\'Identical image has conflicting head labels; manual correction required.\')\n                union(i,previous);exact_removed.add(i)\n            else:digests[r[\'sha256\']]=i\n            for j,h in visual:\n                if (r[\'dhash\']^h).bit_count()<=3:\n                    union(i,j);near_pairs+=1\n            visual.append((i,r[\'dhash\']))\n        components=defaultdict(list)\n        for i,r in enumerate(records):components[find(i)].append(i)\n        assignments={};forced={};conflicts=[]\n        for key,indices in components.items():\n            fs={records[i][\'family\'] for i in indices}\n            if fs & {\'kitchen_cctv\',\'office_camera\'}:\n                split=\'train\';forced[key]=True\n                if \'food_production_media\' in fs:conflicts.append(\'Food-production family connected to training sources by visual/hash similarity; retained in train.\')\n            elif \'food_production_media\' in fs:split=\'test\';forced[key]=True\n            else:\n                signature=\'|\'.join(sorted(records[i][\'member\'] for i in indices))\n                value=int(hashlib.sha256((\'42:\'+signature).encode()).hexdigest()[:8],16)%100\n                split=\'val\' if value<50 else \'test\' if value<80 else \'train\'\n            assignments[key]=split\n        # Ensure each split contains both classes using entire unforced components, never frame-level moves.\n        for _ in range(8):\n            counts={s:[0,0] for s in (\'train\',\'val\',\'test\')}\n            for k,indices in components.items():\n                for i in indices:\n                    if i not in exact_removed:\n                        for c in (0,1):counts[assignments[k]][c]+=records[i][\'counts\'][c]\n            missing=[(s,c) for s in counts for c in (0,1) if not counts[s][c]]\n            if not missing:break\n            s,c=missing[0]\n            candidates=[k for k,indices in components.items() if k not in forced and assignments[k]!=s and any(records[i][\'counts\'][c] for i in indices)]\n            if not candidates:raise ValueError(\'Insufficient independent groups for both classes in every split. Supply a reviewed source mapping.\')\n            chosen=min(candidates,key=lambda k:(len(components[k]),min(records[i][\'member\'] for i in components[k])))\n            assignments[chosen]=s\n        else:raise ValueError(\'Could not build class-covered independent splits.\')\n        dest.mkdir(parents=True,exist_ok=True);manifest=[]\n        for k,indices in components.items():\n            split=assignments[k]\n            for i in indices:\n                if i in exact_removed:continue\n                r=records[i];stem=r[\'sha256\'][:24]\n                image=dest/split/\'images\'/(stem+Path(r[\'member\']).suffix.lower());label=dest/split/\'labels\'/(stem+\'.txt\')\n                image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n                image.write_bytes(z.read(r[\'member\']));label.write_text(\'\\n\'.join(r[\'labels\'])+\'\\n\')\n                manifest.append({key:r[key] for key in (\'member\',\'source_split\',\'family\',\'sha256\',\'counts\')}|{\'split\':split,\'component\':str(k)})\n        for name in z.namelist():\n            if Path(name).name in (\'README.dataset.txt\',\'README.roboflow.txt\'):(dest/Path(name).name).write_bytes(z.read(name))\n    cfg=dest/\'data.yaml\';cfg.write_text(\'path: \'+dest.as_posix()+\'\\ntrain: train/images\\nval: val/images\\ntest: test/images\\nnames: [hairnet, no_hairnet]\\n\')\n    from train import audit\n    report=audit(cfg)\n    report.update(source_dataset=\'KitchenHygiene v3; CC BY 4.0\',source_images=total,excluded=dict(excluded),valid_non_head_polygon_lines=polygon_lines,\n        exact_duplicates_removed=len(exact_removed),near_duplicate_links=near_pairs,source_components=len(components),split_method=\'Seed-42 source families plus mirrored dHash radius-3 components; known kitchen CCTV and office families in train; food-production media held out unless connected to train\',conflicts=conflicts,\n        limitations=[\'Mixed kitchen, office, food-production and stock imagery; not an all-kitchen benchmark\',\'Source identification is heuristic and incomplete\',\'Known kitchen CCTV retained entirely in train; held-out set is not independent college-kitchen footage\',\'Near-duplicate hashing can overmerge sources and cannot guarantee independence\',\'Both public datasets may share images; start from general YOLO weights, not the old fine-tuned checkpoint\'])\n    (dest/\'audit.json\').write_text(json.dumps(report,indent=2))\n    (dest/\'split-manifest.json\').write_text(json.dumps(manifest,indent=2))\n    print(json.dumps(report,indent=2),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'archive\');p.add_argument(\'destination\');a=p.parse_args();prepare(a.archive,a.destination)\n')
print('Training and preparation code ready')


In [ ]:
from google.colab import files
import zipfile,yaml
uploaded=files.upload()
archives=[n for n in uploaded if n.lower().endswith('.zip')]
assert len(archives)==1, 'Select exactly one dataset ZIP; no project ZIP is needed'
archive=Path(archives[0]).resolve()
with zipfile.ZipFile(archive) as z:
    configs=[n for n in z.namelist() if Path(n).name=='data.yaml']
    assert len(configs)==1, 'Expected one dataset configuration'
    configuration=yaml.safe_load(z.read(configs[0]));names=configuration['names']
    if isinstance(names,dict):names=list(names.values())
    prepared=names==['hairnet','no_hairnet']
    if prepared:
        target=Path('/content/kitchen-dataset');target.mkdir(exist_ok=True)
        assert all((target/n.filename).resolve().is_relative_to(target.resolve()) for n in z.infolist()), 'Unsafe ZIP path'
        z.extractall(target)
        data=(target/configs[0]).parent
if not prepared:
    data=Path('/content/kitchenhygiene-heads')
    subprocess.check_call([sys.executable,str(stage/'prepare_kitchenhygiene.py'),str(archive),str(data)])
config=data/'data.yaml'
config.write_text(f'path: {data.as_posix()}\ntrain: train/images\nval: val/images\ntest: test/images\nnames: [hairnet, no_hairnet]\n')
subprocess.check_call([sys.executable,str(stage/'train.py'),'--data',str(config),'--audit-only','--output','/content/kitchenhygiene-results'])
print('Dataset prepared and audited. The next cell starts training.')


In [ ]:
# Start from general YOLO weights, not the old fine-tuned checkpoint.
# Checkpoint selection uses validation data only; test evaluation runs after training.
subprocess.check_call([sys.executable,str(stage/'train.py'),'--data',str(config),
    '--device','0','--epochs','50','--batch','8','--imgsz','640',
    '--patience','15','--class-power','0.25',
    '--output','/content/kitchenhygiene-results'],cwd=stage)


In [ ]:
import json,shutil
results=Path('/content/kitchenhygiene-results')
card=json.loads((results/'model_card.json').read_text())
print('Overall test metrics:',json.dumps(card['test_metrics'],indent=2))
print('Per-class test metrics:',json.dumps(card.get('per_class_test_metrics',{}),indent=2))
best=Path(card['weights']);assert best.exists(), 'Training checkpoint missing'
shutil.copy2(best,results/'hairnet.pt')
for filename in ['audit.json','split-manifest.json','README.dataset.txt','README.roboflow.txt']:
    if (data/filename).exists():shutil.copy2(data/filename,results/filename)
shutil.make_archive('/content/kitchenhygiene-trained-model','zip',results)
files.download('/content/kitchenhygiene-trained-model.zip')


## Return the result
Send **kitchenhygiene-trained-model.zip** back to the chat. It contains actual trained weights, per-class results and provenance. The existing app's old weights should only be replaced after checking these results. A successful notebook run does not certify kitchen safety, and better performance is not guaranteed.
